In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"
os.environ['MKL_THREADING_LAYER'] = "GNU"

In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 42
        environment_string = "cart_pole"
        gold_timesteps =4_000_000
        training_timesteps = 250_000
        num_concepts_selected = 24
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = False
        run_iterative = False 
        run_two_stage = True   
        run_imperfect=False 
        run_intervention=False 
        # Experiment #3
        cbm_accuracy_by_concept = None 
        intervention_probability = 0
        intervention_accuracy_by_concept = None 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 0
        selections_per_round = 0
        initial_concepts = 0
        out_folder = "llm"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--gold_timesteps', help='Number of training timesteps without concepts', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--intervention_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the two stage?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the iterative?', action='store_true')
        parser.add_argument('--run_intervention', help='Run the intervention?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--intervention_probability', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        gold_timesteps = args.gold_timesteps
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        run_intervention = args.run_intervention
        intervention_probability = args.intervention_probability
        intervention_accuracy_by_concept = args.intervention_accuracy_by_concept
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [6]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'gold_timesteps': gold_timesteps,
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
                'run_intervention': run_intervention,
                'run_imperfect': run_imperfect, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'cart_pole', 'training_timesteps': 250000, 'gold_timesteps': 4000000, 'selection_function': 'q_value', 'num_concepts_selected': 24, 'cbm_accuracy_by_concept': None, 'cbm_std_by_concept': None, 'intervention_probability': 0, 'intervention_accuracy_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 0, 'selections_per_round': 0, 'initial_concepts': 0, 'run_basic': False, 'run_iterative': False, 'run_two_stage': True, 'run_intervention': False, 'run_imperfect': False}


In [7]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

### Basic Setup

In [8]:
if is_main:
    concept_list = get_concepts(environment_string,concept_source,seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env, additional_info = get_environment(environment_string, None, seed)   

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [9]:
if is_main:
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    
    if os.path.exists(model_name):
        print("Model exists!")
        groundtruth_model = PPO.load(model_name)
        additional_info['subset_concepts'] = get_concepts(environment_string,"human_selected",seed)
    else:
        if "cyclic" in environment_string or "tree" in environment_string or "glucose" in environment_string:
            policy = "MlpPolicy"
        else:
            policy = "CnnPolicy"
        
        if environment_string == "mimic":
            additional_info['subset_concepts'] = get_concepts(environment_string,"human_selected",seed)
            groundtruth_model = train_ppo_model(ground_truth_env,"mimic_raw",total_timesteps=gold_timesteps,policy=policy,)
        else:
            groundtruth_model = train_ppo_model(ground_truth_env,environment_string,total_timesteps=gold_timesteps,policy=policy)
        groundtruth_model.save(model_name)

Model exists!


In [10]:
height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()
else:
    concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,groundtruth_model,concept_list,list(range(len(concept_list))),environment_string,epochs=25,max_episode_length=10_000)
    torch.save(concept_predictor.state_dict(), model_name)
    concept_predictor.eval()

ALSA lib pulse.c:242:(pulse_connect) ALSA lib pulse.c:242:(pulse_connect) ALSA lib pulse.c:242:(pulse_connect) ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated
PulseAudio: Unable to connect: Connection terminated
PulseAudio: Unable to connect: Connection terminated



PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated




✅ Collection complete!
X shape: (59885, 4, 160, 240), dtype: uint8, size: 8772.2 MB
Y shape: (59885, 4), dtype: float32
Y contains 4 features per sample
X_data shape: (59885, 4, 160, 240), dtype: uint8
Y_data shape: (59885, 12), dtype: float32
📉 Epoch 1/25 | Train Loss 0.4812 | Val Loss 0.3363 | Val F1 0.8517 | Took 23.11s
📉 Epoch 2/25 | Train Loss 0.3721 | Val Loss 0.3023 | Val F1 0.8699 | Took 23.62s
📉 Epoch 3/25 | Train Loss 0.3225 | Val Loss 0.2802 | Val F1 0.8762 | Took 23.33s
📉 Epoch 4/25 | Train Loss 0.2812 | Val Loss 0.2648 | Val F1 0.8865 | Took 23.65s
📉 Epoch 5/25 | Train Loss 0.2723 | Val Loss 0.2615 | Val F1 0.8895 | Took 23.69s
📉 Epoch 6/25 | Train Loss 0.2578 | Val Loss 0.2507 | Val F1 0.8896 | Took 23.42s
📉 Epoch 7/25 | Train Loss 0.2520 | Val Loss 0.2488 | Val F1 0.8913 | Took 23.90s
📉 Epoch 8/25 | Train Loss 0.2459 | Val Loss 0.2403 | Val F1 0.8965 | Took 23.65s
📉 Epoch 9/25 | Train Loss 0.2400 | Val Loss 0.2381 | Val F1 0.8976 | Took 24.26s
📉 Epoch 10/25 | Train Loss

In [17]:
if is_main:
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, concept_list, seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=list(range(len(concept_list))))   


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [18]:
if is_main:
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=300_000,custom_name="{}_two_stage_greedy".format(environment_string))   

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated

ALSA lib pulse.c:242:(pulse_connect) PulseAudio: Unable to connect: Connection terminated



approx_kl,█▇▅▃▂▁▃▄▁▁▂▂▃▃▃▃▃▂▁▂▃▃▄▅▃▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂
clip_fraction,█▃▃▃▃▁▁▂▂▂▄▄▁▁▄▃▃▃▃▄▄▄▁▂▂▃▅▆▅▂▁▂▁▃▂▁▂▂▅▅
ema_norm_reward,▁▁▁▁▂▄▄▅▅▅▅▅▆▆▆▆▆▅██▅▅▅▅▆▆▆▆▆▇▇▆▇▆▇▇▇▇▇▇
entropy_loss,▁▂▂▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▆▇▇▇▇▇▇█▇▇█████████▇
episode_length_mean,▁▁▁▁▁▂▂▃▂▃▃▃▃▃▂▃▄▄▂▄▃▃▃▃▂▃▃▅▆▆▄▆▃▃▇▃▄█▅▄
episode_reward_max,▃▁▃▂▃▅▅▅▃▂▆▅▁▅▅▃▆▄▃▃▅▆▆▃▅▇▃▃█▄▅▆▄▄▆▄▄▅▄▃
episode_reward_mean,▁▁▁▁▁▁▁▃▂▂▂▅▂▃▂▃▄▄▃▃▃▃▂▄▄▄▆▇▃█▃▅▄▃▄▃▅▄▅▄
episode_reward_min,▁▁▁▁▁▁▃▂▁▄▅▆▄▃▃▄▃▄█▄▄▄▃▂▃▃▃▄█▄▅▄▆▄▅▆▄▄▄▅
episodes_completed,▁▆█▆▆▁▁▁▁▁▃▁▁▁▆▁▁▁▁▁▆▁▃▃▁▃▁▁▁▁▁▁▁▁▁▁▃▁▃▁
explained_variance,▁▂▂▂▂▁▂▂▂▂▂▁▂▃▂▃▂▃▃▄▇▇▇███▇▆▆▅▅████▆▅▆▆▅
+1,...
